# K-Means Clustering

**Companion lesson:** https://ml-viz.vercel.app/courses/clustering/01-k-means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## K-Means Algorithm

1. Initialize centroids randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat until convergence

In [ ]:
np.random.seed(42)
K = 3
centers = np.array([[0, 0], [5, 0], [2.5, 4.3]])
n_per = 50
X = np.vstack([np.random.randn(n_per, 2) * 0.8 + c for c in centers])

def kmeans(X, K, n_iter=10):
    centroids = X[np.random.choice(len(X), K, replace=False)]
    history = [centroids.copy()]
    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None], axis=2)
        labels = np.argmin(dists, axis=1)
        centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])
        history.append(centroids.copy())
    return labels, centroids, history

labels, centroids, history = kmeans(X, K)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    step_centroids = history[i]
    dists = np.linalg.norm(X[:, None] - step_centroids[None], axis=2)
    step_labels = np.argmin(dists, axis=1)
    for k in range(K):
        mask = step_labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=['#818cf8', '#14b8a6', '#eab308'][k], s=10, alpha=0.5)
    ax.scatter(step_centroids[:, 0], step_centroids[:, 1], c='white', s=200, marker='*', edgecolors='black', linewidths=1)
    ax.set_title(f'Iteration {i}', color='white', fontsize=11)
    ax.set_xlim(-2, 7)
    ax.set_ylim(-2, 6)
    ax.set_aspect('equal')
    ax.axis('off')
plt.suptitle('K-Means Convergence', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Elbow Method

Plot inertia vs K — look for the "elbow" where improvement slows.

In [ ]:
inertias = []
for k in range(1, 10):
    _, cents, _ = kmeans(X, k)
    dists = np.linalg.norm(X[:, None] - cents[None], axis=2)
    labels = np.argmin(dists, axis=1)
    inertia = sum(np.sum((X[labels == k] - cents[k])**2) for k in range(k))
    inertias.append(inertia)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, 10), inertias, 'o-', color='#818cf8', linewidth=2, markersize=8)
axes[0].axvline(3, color='#f43f5e', linestyle='--', alpha=0.7, label='Elbow at K=3')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (Within-Cluster SS)')
axes[0].set_title('Elbow Method', color='white')
axes[0].legend()

silhouette_scores = []
for k in range(2, 10):
    _, cents, _ = kmeans(X, k)
    dists = np.linalg.norm(X[:, None] - cents[None], axis=2)
    labels = np.argmin(dists, axis=1)
    s = 0
    for i in range(len(X)):
        ci = labels[i]
        a = np.mean(np.linalg.norm(X[labels == ci] - X[i], axis=1))
        b = min(np.mean(np.linalg.norm(X[labels == j] - X[i], axis=1)) for j in range(k) if j != ci)
        s += (b - a) / max(a, b)
    silhouette_scores.append(s / len(X))
axes[1].plot(range(2, 10), silhouette_scores, 's-', color='#14b8a6', linewidth=2, markersize=8)
axes[1].axvline(3, color='#f43f5e', linestyle='--', alpha=0.7, label='Best at K=3')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', color='white')
axes[1].legend()
plt.tight_layout()
plt.show()